In [ ]:
import glob
import json
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid")

In [ ]:
with open("../../data/accession_translator.json", "r") as s:
    accession_translator = json.load(s)

In [ ]:
gffs = []
paths = glob.glob("../../intermediates/benchmarking_results/geneml/*.gff3")
for path in paths:
    accession = path.split('/')[-1].replace(".gff3", "")
    gff = pd.read_csv(path, sep="\t", comment="#", header=None, names=[
        "seqid", "source", "type", "start", "end", "score", "strand", "phase", "attributes"])
    gff["accession"] = accession
    gff["genome"] = accession_translator[accession]
    gff["genome"] = pd.Categorical(gff["genome"], categories=list(accession_translator.values()), ordered=True)
    gffs.append(gff)
all_gff = pd.concat(gffs, ignore_index=True)

In [ ]:
transcripts = all_gff[all_gff["type"] == "mRNA"].copy()
attributes_split = transcripts['attributes'].str.split(';', expand=True).stack().str.split('=', expand=True)
attributes_split.columns = ['key', 'value']
attributes_pivot = attributes_split.pivot_table(index=attributes_split.index.get_level_values(0), columns='key', values='value', aggfunc='first')
transcripts = pd.concat([transcripts.reset_index(drop=True), attributes_pivot.reset_index(drop=True)], axis=1)


In [ ]:
# Distance from PRIMARY to all non-PRIMARY transcript boundaries (strand-aware)
primary_coords = (
    transcripts.loc[transcripts['TranscriptVariant'] == 'PRIMARY', ['genome', 'Parent', 'start', 'end']]
    .rename(columns={'start': 'primary_start', 'end': 'primary_end'})
)

non_primary = transcripts.loc[
    transcripts['TranscriptVariant'] != 'PRIMARY',
    ['genome', 'Parent', 'start', 'end', 'seqid', 'strand', 'TranscriptVariant']
].rename(columns={'start': 'transcript_start', 'end': 'transcript_end'})

transcript_distances = non_primary.merge(
    primary_coords,
    on=['genome', 'Parent'],
    how='inner'
)

transcript_distances['distance'] = (
    transcript_distances['transcript_start'] - transcript_distances['primary_start']
)

reverse_mask = transcript_distances['strand'] == '-'
transcript_distances.loc[reverse_mask, 'distance'] = (
    transcript_distances.loc[reverse_mask, 'primary_end']
    - transcript_distances.loc[reverse_mask, 'transcript_end']
)

transcript_distances = transcript_distances[transcript_distances['distance'] != 0]


In [ ]:
transcript_distances.groupby('TranscriptVariant', observed=True).size().rename('count').reset_index().sort_values('count', ascending=False)

In [ ]:
# Cumulative coverage curve (empirical CDF, no binning)
non_negative = transcript_distances[transcript_distances['distance'] >= 0]['distance']
sorted_distances = non_negative.sort_values().values
cumulative_pct = 100 * (pd.Series(range(1, len(sorted_distances) + 1)) / len(sorted_distances)).values

target_distance = 120
idx = (sorted_distances <= target_distance).sum() - 1
coverage_at_target = cumulative_pct[idx] if idx >= 0 else 0

fig, ax = plt.subplots(figsize=(10, 4))

ax.plot(sorted_distances, cumulative_pct, color='black', linewidth=1.5)
ax.fill_between(sorted_distances, cumulative_pct, alpha=0.1, color='black')

ax.set_xlabel('Distance downstream of primary start site (bp)', fontsize=12)
ax.set_ylabel('Cumulative % of transcripts', fontsize=12)
ax.set_xlim(0, 1000)
ax.set_ylim(0, 100)
ax.set_xticks(range(0, 1001, 100))
ax.xaxis.set_tick_params(rotation=45, colors='black')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v:.0f}%'))
ax.tick_params(axis='y', colors='black')

for spine in ax.spines.values():
    spine.set_color('black')

ax.axvline(target_distance, color='black', linewidth=0.6, linestyle='--')
ax.axhline(coverage_at_target, color='black', linewidth=0.6, linestyle='--')
ax.annotate(
    f'{coverage_at_target:.0f}% @ {target_distance} bp',
    xy=(target_distance, coverage_at_target),
    xytext=(target_distance + 10, max(coverage_at_target - 4, 2)),
    fontsize=8,
    color='black',
)

plt.tight_layout()
fig.savefig("../../figures/Figure_4B.svg", bbox_inches="tight")
plt.show()

In [ ]:
# Count splicing types per parent within each genome (observed pairs only)
parent_splicing_counts = (
    transcripts
    .groupby(['genome', 'Parent', 'TranscriptVariant'], observed=True)
    .size()
    .unstack(fill_value=0)
)

assert (parent_splicing_counts['PRIMARY'] == 1).all()

In [ ]:
splice_types = {
    "ALT_FIRST_EXON" : "Alt First Exon",
    "ALT_LAST_EXON" : "Alt Last Exon",
    "INTRON_RETENTION" : "Intron Retention",
    "EXON_SKIPPING" : "Exon Skipping",
    "ALT_5_SPLICE_SITE" : "Alt 5' Splice Site",
    "ALT_3_SPLICE_SITE" : "Alt 3' Splice Site",
    "COMPLEX" : "Complex",
}

In [ ]:
# Prevalence per genome from parent_splicing_counts: fraction of parents with count >= 1 for each variant
parent_counts_ordered = parent_splicing_counts.reindex(columns=list(splice_types.keys()), fill_value=0)

prevalence_df = (
    parent_counts_ordered[list(splice_types.keys())]
    .ge(1)
    .groupby(level='genome', observed=True)
    .mean()
    .stack()
    .rename('prevalence')
    .reset_index()
    .rename(columns={'level_1': 'TranscriptVariant'})
)

prevalence_df['TranscriptVariant'] = prevalence_df['TranscriptVariant'].map(splice_types)
prevalence_df['TranscriptVariant'] = pd.Categorical(
    prevalence_df['TranscriptVariant'],
    categories=list(splice_types.values()),
    ordered=True,
)

In [ ]:
# Plot splice type ratios for each genome as a heatmap
heatmap_data = prevalence_df.pivot(index='TranscriptVariant', columns='genome', values='prevalence')
heatmap_data['Average'] = heatmap_data.mean(axis=1)

regular_cols = [col for col in heatmap_data.columns if col != 'Average']
regular_data = heatmap_data[regular_cols]
average_col_data = heatmap_data[['Average']]
total_row_data = pd.DataFrame([regular_data.sum(axis=0)], index=['Total'])
total_avg_data = pd.DataFrame([[average_col_data['Average'].sum()]], index=['Total'], columns=['Average'])

vmin = 0
vmax = 0.2

fig = plt.figure(figsize=(12, 9))
gs = fig.add_gridspec(
    3,
    5,
    width_ratios=[max(len(regular_cols), 1), 0.10, 1, 0.28, 0.18],
    height_ratios=[max(len(regular_data.index), 1), 0, 1],
    wspace=0.02,
    hspace=0,
)

ax_main = fig.add_subplot(gs[0, 0])
ax_row_spacer = fig.add_subplot(gs[1, 0])
ax_row_total = fig.add_subplot(gs[2, 0], sharex=ax_main)
ax_col_spacer = fig.add_subplot(gs[:, 1])
ax_avg = fig.add_subplot(gs[0, 2])
ax_empty_mid = fig.add_subplot(gs[1, 2])
ax_total_avg = fig.add_subplot(gs[2, 2])
ax_cbar_spacer = fig.add_subplot(gs[:, 3])
ax_cbar = fig.add_subplot(gs[0, 4])
ax_cbar_mid = fig.add_subplot(gs[1, 4])
ax_cbar_bottom = fig.add_subplot(gs[2, 4])

sns.heatmap(
    regular_data,
    annot=regular_data * 100,
    fmt='.1f',
    cmap='Blues',
    vmin=vmin,
    vmax=vmax,
    cbar=False,
    ax=ax_main,
)

ax_row_spacer.axis('off')

sns.heatmap(
    total_row_data,
    annot=total_row_data * 100,
    fmt='.1f',
    cmap=sns.color_palette(['white'], as_cmap=True),
    cbar=False,
    annot_kws={'color': 'black'},
    ax=ax_row_total,
)

ax_col_spacer.axis('off')
ax_empty_mid.axis('off')
ax_cbar_spacer.axis('off')
ax_cbar_mid.axis('off')
ax_cbar_bottom.axis('off')

hm_avg = sns.heatmap(
    average_col_data,
    annot=average_col_data * 100,
    fmt='.1f',
    cmap='Blues',
    vmin=vmin,
    vmax=vmax,
    yticklabels=False,
    cbar=False,
    ax=ax_avg,
)

sns.heatmap(
    total_avg_data,
    annot=total_avg_data * 100,
    fmt='.1f',
    cmap=sns.color_palette(['white'], as_cmap=True),
    cbar=False,
    annot_kws={'color': 'black'},
    yticklabels=False,
    ax=ax_total_avg,
)

cbar = fig.colorbar(hm_avg.collections[0], cax=ax_cbar, ticks=[0, 0.05, 0.10, 0.15, 0.20])
cbar.set_label('Prevalence (%)', fontsize=11, labelpad=4)
cbar.ax.yaxis.set_label_position('left')
cbar.ax.tick_params(labelsize=11)
cbar.ax.set_yticklabels(['0', '5', '10', '15', '20'])

ax_main.set_xlabel('')
ax_main.set_ylabel('Transcript variant', fontsize=14, labelpad=15)
ax_row_total.set_xlabel('Genome', fontsize=14)
ax_row_total.set_ylabel('', fontsize=14)
ax_avg.set_xlabel('', fontsize=14)
ax_avg.set_ylabel('', fontsize=14)
ax_total_avg.set_xlabel('', fontsize=14)
ax_total_avg.set_ylabel('', fontsize=14)

ax_main.tick_params(axis='x', bottom=False, labelbottom=False)
ax_row_total.tick_params(axis='x', pad=-5)
ax_avg.tick_params(axis='x', pad=55)
ax_total_avg.tick_params(axis='x', bottom=False, labelbottom=False)
ax_row_total.set_yticklabels(ax_row_total.get_yticklabels(), rotation=0)

for label in ax_row_total.get_xticklabels():
    label.set_rotation(45)
    label.set_horizontalalignment('right')
    label.set_rotation_mode('anchor')

for label in ax_avg.get_xticklabels():
    label.set_rotation(45)
    label.set_horizontalalignment('right')
    label.set_rotation_mode('anchor')

fig.subplots_adjust(left=0.16, right=0.97, bottom=0.22, top=0.96)
pos = ax_cbar.get_position()
ax_cbar.set_position([pos.x0, pos.y0 + 0.08, pos.width, pos.height - 0.16])

fig.savefig("../../figures/Figure_4A.svg", bbox_inches="tight", pad_inches=0.03)
plt.show()